# 🤖 Agente Text-to-SQL com LLM

Agente inteligente que converte perguntas em linguagem natural para consultas SQL,
executa no banco de dados e responde em português.

**Tecnologias:** Groq (LLaMA 3), SQLite, Python

In [1]:
# Instalação das dependências
!pip install groq -q

print("Dependências instaladas!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 6.0 MB/s eta 0:00:00
Dependências instaladas!


In [2]:
import os
import sqlite3
from groq import Groq

# Configuração da API Key
os.environ["GROQ_API_KEY"] = "chave-oculta"

# Inicializa o cliente Groq
cliente = Groq(api_key=os.environ["GROQ_API_KEY"])

print("Cliente Groq configurado!")

Cliente Groq configurado!


In [3]:
# Cria o banco de dados em memória
conn = sqlite3.connect("vendas.db")
cursor = conn.cursor()

# Cria as tabelas
cursor.executescript("""
    CREATE TABLE IF NOT EXISTS clientes (
        id INTEGER PRIMARY KEY,
        nome TEXT,
        cidade TEXT,
        estado TEXT
    );

    CREATE TABLE IF NOT EXISTS produtos (
        id INTEGER PRIMARY KEY,
        nome TEXT,
        categoria TEXT,
        preco REAL
    );

    CREATE TABLE IF NOT EXISTS pedidos (
        id INTEGER PRIMARY KEY,
        cliente_id INTEGER,
        produto_id INTEGER,
        quantidade INTEGER,
        data TEXT,
        FOREIGN KEY (cliente_id) REFERENCES clientes(id),
        FOREIGN KEY (produto_id) REFERENCES produtos(id)
    );
""")

# Insere dados fictícios
cursor.executescript("""
    INSERT OR IGNORE INTO clientes VALUES
        (1, 'João Silva', 'São Paulo', 'SP'),
        (2, 'Maria Souza', 'Belo Horizonte', 'MG'),
        (3, 'Carlos Lima', 'Franca', 'SP'),
        (4, 'Ana Costa', 'Rio de Janeiro', 'RJ'),
        (5, 'Pedro Alves', 'Uberlândia', 'MG');

    INSERT OR IGNORE INTO produtos VALUES
        (1, 'Notebook', 'Eletrônicos', 3500.00),
        (2, 'Mouse', 'Eletrônicos', 150.00),
        (3, 'Cadeira Gamer', 'Móveis', 1200.00),
        (4, 'Monitor', 'Eletrônicos', 2000.00),
        (5, 'Teclado', 'Eletrônicos', 300.00);

    INSERT OR IGNORE INTO pedidos VALUES
        (1, 1, 1, 2, '2024-01-15'),
        (2, 2, 3, 1, '2024-02-20'),
        (3, 3, 2, 5, '2024-03-10'),
        (4, 1, 4, 1, '2024-03-22'),
        (5, 4, 5, 3, '2024-04-05'),
        (6, 5, 1, 1, '2024-04-18'),
        (7, 2, 2, 2, '2024-05-01'),
        (8, 3, 4, 2, '2024-05-15'),
        (9, 4, 3, 1, '2024-06-10'),
        (10, 5, 5, 4, '2024-06-25');
""")

conn.commit()
print("Banco de dados criado com sucesso!")

Banco de dados criado com sucesso!


In [8]:
def gerar_sql(pergunta: str) -> str:
    """
    Recebe uma pergunta em português e retorna o SQL correspondente.
    """

    # Contexto do banco de dados que enviamos para o LLM
    schema = """
    Você tem acesso a um banco de dados SQLite com as seguintes tabelas:

    clientes (id, nome, cidade, estado)
    produtos (id, nome, categoria, preco)
    pedidos (id, cliente_id, produto_id, quantidade, data)

    Regras:
    - Responda APENAS com o SQL, sem explicações, sem markdown, sem comentários
    - Use sempre JOIN quando precisar cruzar tabelas
    - O campo data está no formato YYYY-MM-DD
    - Nunca use DELETE, DROP ou UPDATE
    """

    resposta = cliente.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": schema},
            {"role": "user", "content": f"Gere o SQL para: {pergunta}"}
        ],
        temperature=0  # zero = mais determinístico, menos criativo
    )

    return resposta.choices[0].message.content.strip()

print("Função gerar_sql criada!")

Função gerar_sql criada!


In [5]:
def executar_sql(sql: str) -> list:
    """
    Recebe um SQL, executa no banco de dados e retorna os resultados.
    """
    try:
        cursor.execute(sql)
        resultados = cursor.fetchall()
        colunas = [descricao[0] for descricao in cursor.description]
        return {"colunas": colunas, "dados": resultados}

    except Exception as erro:
        return {"erro": str(erro)}

print("Função executar_sql criada!")

Função executar_sql criada!


In [9]:
def agente(pergunta: str) -> str:
    """
    Função principal do agente Text-to-SQL.
    Recebe uma pergunta em português e retorna a resposta em português.
    """

    print(f"📝 Pergunta: {pergunta}")

    # Passo 1: gera o SQL com o LLM
    sql = gerar_sql(pergunta)
    print(f"🔍 SQL gerado: {sql}")

    # Passo 2: executa o SQL no banco
    resultado = executar_sql(sql)

    if "erro" in resultado:
        return f"Erro ao executar a consulta: {resultado['erro']}"

    # Passo 3: manda os resultados para o LLM responder em português
    resposta = cliente.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "system",
                "content": "Você é um assistente de dados. Responda sempre em português de forma clara e objetiva."
            },
            {
                "role": "user",
                "content": f"""
                O usuário perguntou: {pergunta}
                Os dados retornados foram: {resultado}
                Responda a pergunta do usuário com base nesses dados.
                """
            }
        ]
    )

    return resposta.choices[0].message.content.strip()

print("Agente criado!")

Agente criado!


In [10]:
# Teste 1 — pergunta simples
resposta = agente("Quais são os clientes cadastrados?")
print(resposta)

📝 Pergunta: Quais são os clientes cadastrados?
🔍 SQL gerado: SELECT * FROM clientes
Os clientes cadastrados são:

1. João Silva, de São Paulo, SP
2. Maria Souza, de Belo Horizonte, MG
3. Carlos Lima, de Franca, SP
4. Ana Costa, de Rio de Janeiro, RJ
5. Pedro Alves, de Uberlândia, MG

Esses são os 5 clientes registrados no sistema, com suas respectivas informações de nome, cidade e estado.


In [11]:
# Teste 2 — pergunta com cruzamento de tabelas
resposta = agente("Qual foi o produto mais vendido?")
print(resposta)

📝 Pergunta: Qual foi o produto mais vendido?
🔍 SQL gerado: SELECT p.nome 
FROM pedidos pe 
JOIN produtos p ON pe.produto_id = p.id 
GROUP BY p.nome 
ORDER BY SUM(pe.quantidade) DESC 
LIMIT 1
O produto mais vendido foi o "Teclado".


In [12]:
# Teste 3 — pergunta com filtro e cálculo
resposta = agente("Qual o valor total gasto por cada cliente?")
print(resposta)

📝 Pergunta: Qual o valor total gasto por cada cliente?
🔍 SQL gerado: SELECT c.nome, SUM(p.preco * pe.quantidade) AS total_gasto 
FROM clientes c 
JOIN pedidos pe ON c.id = pe.cliente_id 
JOIN produtos p ON pe.produto_id = p.id 
GROUP BY c.nome
Com base nos dados fornecidos, podemos responder à pergunta do usuário da seguinte forma:

O valor total gasto por cada cliente é o seguinte:

- Ana Costa: R$ 2100,00
- Carlos Lima: R$ 4750,00
- João Silva: R$ 9000,00
- Maria Souza: R$ 1500,00
- Pedro Alves: R$ 4700,00

Esses valores são extraídos diretamente da lista de dados retornada, onde cada cliente é associado ao seu respectivo "total_gasto".


In [13]:
# Recria o banco com dados mais ricos
conn = sqlite3.connect("vendas.db")
cursor = conn.cursor()

cursor.executescript("""
    DROP TABLE IF EXISTS pedidos;
    DROP TABLE IF EXISTS produtos;
    DROP TABLE IF EXISTS clientes;

    CREATE TABLE clientes (
        id INTEGER PRIMARY KEY,
        nome TEXT,
        cidade TEXT,
        estado TEXT,
        segmento TEXT
    );

    CREATE TABLE produtos (
        id INTEGER PRIMARY KEY,
        nome TEXT,
        categoria TEXT,
        preco REAL
    );

    CREATE TABLE pedidos (
        id INTEGER PRIMARY KEY,
        cliente_id INTEGER,
        produto_id INTEGER,
        quantidade INTEGER,
        data TEXT,
        FOREIGN KEY (cliente_id) REFERENCES clientes(id),
        FOREIGN KEY (produto_id) REFERENCES produtos(id)
    );

    INSERT INTO clientes VALUES
        (1,  'João Silva',      'São Paulo',       'SP', 'Varejo'),
        (2,  'Maria Souza',     'Belo Horizonte',  'MG', 'Corporativo'),
        (3,  'Carlos Lima',     'Franca',          'SP', 'Varejo'),
        (4,  'Ana Costa',       'Rio de Janeiro',  'RJ', 'Corporativo'),
        (5,  'Pedro Alves',     'Uberlândia',      'MG', 'Varejo'),
        (6,  'Lucia Mendes',    'Curitiba',        'PR', 'Corporativo'),
        (7,  'Rafael Torres',   'Porto Alegre',    'RS', 'Varejo'),
        (8,  'Fernanda Rocha',  'Salvador',        'BA', 'Varejo'),
        (9,  'Bruno Carvalho',  'Fortaleza',       'CE', 'Corporativo'),
        (10, 'Juliana Nunes',   'Brasília',        'DF', 'Corporativo');

    INSERT INTO produtos VALUES
        (1,  'Notebook',         'Eletrônicos',   3500.00),
        (2,  'Mouse',            'Eletrônicos',    150.00),
        (3,  'Cadeira Gamer',    'Móveis',        1200.00),
        (4,  'Monitor',          'Eletrônicos',   2000.00),
        (5,  'Teclado',          'Eletrônicos',    300.00),
        (6,  'Headset',          'Eletrônicos',    450.00),
        (7,  'Webcam',           'Eletrônicos',    350.00),
        (8,  'Mesa Escritório',  'Móveis',         900.00),
        (9,  'Suporte Monitor',  'Móveis',         250.00),
        (10, 'HD Externo',       'Eletrônicos',    400.00);

    INSERT INTO pedidos VALUES
        (1,  1,  1,  1, '2023-01-10'),
        (2,  2,  3,  2, '2023-01-22'),
        (3,  3,  2,  3, '2023-02-05'),
        (4,  4,  4,  1, '2023-02-18'),
        (5,  5,  5,  2, '2023-03-01'),
        (6,  6,  6,  1, '2023-03-14'),
        (7,  7,  7,  2, '2023-04-02'),
        (8,  8,  8,  1, '2023-04-20'),
        (9,  9,  9,  3, '2023-05-08'),
        (10, 10, 10, 2, '2023-05-25'),
        (11, 1,  4,  2, '2023-06-10'),
        (12, 2,  5,  4, '2023-06-28'),
        (13, 3,  6,  1, '2023-07-15'),
        (14, 4,  7,  2, '2023-07-30'),
        (15, 5,  8,  1, '2023-08-12'),
        (16, 6,  1,  1, '2023-08-25'),
        (17, 7,  2,  5, '2023-09-05'),
        (18, 8,  3,  1, '2023-09-20'),
        (19, 9,  4,  1, '2023-10-08'),
        (20, 10, 5,  3, '2023-10-22'),
        (21, 1,  6,  2, '2023-11-05'),
        (22, 2,  7,  1, '2023-11-18'),
        (23, 3,  8,  1, '2023-12-02'),
        (24, 4,  9,  2, '2023-12-15'),
        (25, 5,  10, 1, '2023-12-28'),
        (26, 6,  2,  3, '2024-01-10'),
        (27, 7,  3,  1, '2024-01-25'),
        (28, 8,  4,  2, '2024-02-08'),
        (29, 9,  5,  2, '2024-02-20'),
        (30, 10, 6,  1, '2024-03-05'),
        (31, 1,  7,  1, '2024-03-18'),
        (32, 2,  8,  2, '2024-04-01'),
        (33, 3,  9,  3, '2024-04-15'),
        (34, 4,  10, 1, '2024-04-28'),
        (35, 5,  1,  1, '2024-05-10'),
        (36, 6,  2,  4, '2024-05-22'),
        (37, 7,  3,  1, '2024-06-05'),
        (38, 8,  4,  1, '2024-06-18'),
        (39, 9,  5,  3, '2024-07-02'),
        (40, 10, 6,  2, '2024-07-15');
""")

conn.commit()
print("Banco de dados atualizado com sucesso!")
print(f"Clientes: 10")
print(f"Produtos: 10")
print(f"Pedidos: 40")

Banco de dados atualizado com sucesso!
Clientes: 10
Produtos: 10
Pedidos: 40


In [14]:
# Teste com dados novos — análise por período
resposta = agente("Qual foi o total de vendas em 2023 e em 2024?")
print(resposta)

📝 Pergunta: Qual foi o total de vendas em 2023 e em 2024?
🔍 SQL gerado: SELECT 
    SUM(CASE WHEN STRFTIME('%Y', p.data) = '2023' THEN p.quantidade * pr.preco ELSE 0 END) AS total_2023,
    SUM(CASE WHEN STRFTIME('%Y', p.data) = '2024' THEN p.quantidade * pr.preco ELSE 0 END) AS total_2024
FROM 
    pedidos p
JOIN 
    produtos pr ON p.produto_id = pr.id
Com base nos dados fornecidos, podemos observar que existem valores específicos para o total de vendas em 2023 e 2024. 

O total de vendas em 2023 foi de R$ 31.200,00 e em 2024 foi de R$ 19.100,00.


In [15]:
def agente(pergunta: str) -> str:
    """
    Versão melhorada do agente com tratamento de perguntas inválidas.
    """

    print(f"📝 Pergunta: {pergunta}")

    # Passo 1: valida se a pergunta faz sentido para o banco
    validacao = cliente.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "system",
                "content": """Você valida se uma pergunta pode ser respondida com um banco de dados
                de vendas que contém: clientes, produtos e pedidos.
                Responda APENAS com 'SIM' ou 'NAO'."""
            },
            {
                "role": "user",
                "content": pergunta
            }
        ],
        temperature=0
    )

    pode_responder = validacao.choices[0].message.content.strip().upper()

    if "NAO" in pode_responder:
        return "❌ Essa pergunta está fora do escopo do banco de dados. Tente perguntar sobre clientes, produtos ou pedidos."

    # Passo 2: gera o SQL
    sql = gerar_sql(pergunta)
    print(f"🔍 SQL gerado: {sql}")

    # Passo 3: executa o SQL
    resultado = executar_sql(sql)

    if "erro" in resultado:
        return f"⚠️ Não consegui executar a consulta: {resultado['erro']}"

    if not resultado["dados"]:
        return "🔍 Nenhum dado encontrado para essa pergunta."

    # Passo 4: responde em português
    resposta = cliente.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "system",
                "content": "Você é um assistente de dados. Responda sempre em português de forma clara e objetiva."
            },
            {
                "role": "user",
                "content": f"""
                O usuário perguntou: {pergunta}
                Os dados retornados foram: {resultado}
                Responda a pergunta do usuário com base nesses dados.
                """
            }
        ]
    )

    return resposta.choices[0].message.content.strip()

print("Agente atualizado com tratamento de perguntas inválidas!")

Agente atualizado com tratamento de perguntas inválidas!


In [16]:
# Teste — pergunta fora do escopo
resposta = agente("Qual é a previsão do tempo para amanhã?")
print(resposta)

📝 Pergunta: Qual é a previsão do tempo para amanhã?
❌ Essa pergunta está fora do escopo do banco de dados. Tente perguntar sobre clientes, produtos ou pedidos.


In [17]:
# Histórico de conversa
historico = []

def agente_com_historico(pergunta: str) -> str:
    """
    Versão final do agente com histórico de conversa e guardrail.
    """

    print(f"📝 Pergunta: {pergunta}")

    # Guardrail
    validacao = cliente.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "system",
                "content": """Você valida se uma pergunta pode ser respondida com um banco de dados
                de vendas que contém: clientes, produtos e pedidos.
                Responda APENAS com 'SIM' ou 'NAO'."""
            },
            {"role": "user", "content": pergunta}
        ],
        temperature=0
    )

    if "NAO" in validacao.choices[0].message.content.strip().upper():
        return "❌ Pergunta fora do escopo. Tente perguntar sobre clientes, produtos ou pedidos."

    # Adiciona a pergunta ao histórico
    historico.append({"role": "user", "content": pergunta})

    # Gera o SQL com contexto do histórico
    sql_response = cliente.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "system",
                "content": """
                Você tem acesso a um banco de dados SQLite com as seguintes tabelas:
                clientes (id, nome, cidade, estado, segmento)
                produtos (id, nome, categoria, preco)
                pedidos (id, cliente_id, produto_id, quantidade, data)

                Regras:
                - Responda APENAS com o SQL, sem explicações, sem markdown
                - Use sempre JOIN quando precisar cruzar tabelas
                - O campo data está no formato YYYY-MM-DD
                - Nunca use DELETE, DROP ou UPDATE
                """
            }
        ] + historico,
        temperature=0
    )

    sql = sql_response.choices[0].message.content.strip()
    print(f"🔍 SQL gerado: {sql}")

    # Executa o SQL
    resultado = executar_sql(sql)

    if "erro" in resultado:
        return f"⚠️ Não consegui executar a consulta: {resultado['erro']}"

    if not resultado["dados"]:
        return "🔍 Nenhum dado encontrado."

    # Responde em português
    resposta_final = cliente.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "system",
                "content": "Você é um assistente de dados. Responda sempre em português de forma clara e objetiva."
            },
            {
                "role": "user",
                "content": f"Pergunta: {pergunta}\nDados: {resultado}\nResponda em português."
            }
        ]
    )

    resposta = resposta_final.choices[0].message.content.strip()

    # Adiciona a resposta ao histórico
    historico.append({"role": "assistant", "content": resposta})

    return resposta

print("Agente com histórico criado!")

Agente com histórico criado!


In [18]:
# Teste com histórico — perguntas encadeadas
resposta1 = agente_com_historico("Quais clientes são do estado de MG?")
print(resposta1)
print("\n" + "="*50 + "\n")

resposta2 = agente_com_historico("E quanto eles gastaram no total?")
print(resposta2)

📝 Pergunta: Quais clientes são do estado de MG?
🔍 SQL gerado: SELECT nome FROM clientes WHERE estado = 'MG'
Desculpe, mas não há informações suficientes nos dados fornecidos para determinar quais clientes são do estado de Minas Gerais (MG). Os dados apenas contêm os nomes dos clientes, sem qualquer informação sobre sua localização geográfica, como estado ou cidade.

Para responder à pergunta, precisaríamos de mais informações, como um conjunto de dados que incluísse a coluna "estado" ou "localização" com os respectivos valores. Por exemplo:

{'colunas': ['nome', 'estado'], 'dados': [('Maria Souza', 'MG'), ('Pedro Alves', 'SP')]}

Nesse caso, poderíamos identificar que Maria Souza é do estado de Minas Gerais (MG). No entanto, com os dados atuais, não é possível determinar a localização dos clientes.


📝 Pergunta: E quanto eles gastaram no total?
🔍 SQL gerado: SELECT SUM(p.preco * pe.quantidade) 
FROM clientes c 
JOIN pedidos pe ON c.id = pe.cliente_id 
JOIN produtos p ON pe.produto_id =